# STE + CW Reparameterization — Vocabulary Collapse Fix

**Problem**: v6/v7 self-routing uses `argmax(C @ h)` which collapses to a single token on GSM8K.

**Proposed fix**: Reparameterize as `argmax((CW) @ h)` with frozen orthogonal C and learned W, using STE for gradient flow.

```
Current v6:  h → lm_head → logits[bv:] → argmax → token_id
Proposed:    h → W → Wh  → (C @ Wh)   → multinomial(T) → token_id  ← STE grad → W
```

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from transformers import AutoTokenizer
from sorl.sorl_wrapper import SorlModelWrapper
from data.pt_dataset import get_dataset

device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
MODEL_NAME = "Qwen/Qwen3-0.6B"
N_ABS = 32
K     = 8
D_SHOW = 64

model = SorlModelWrapper.from_pretrained(MODEL_NAME, abstract_vocab_size_list=[N_ABS])
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
model = model.to(device).eval()

bv          = int(model.vocab_sizes[0].item())
d           = model.model.config.hidden_size
model_dtype = next(model.parameters()).dtype
print(f"Model: {MODEL_NAME} | d={d} | bv={bv} | n_abs={N_ABS}")
print(f"Device: {device} | dtype: {model_dtype}")

/Users/fangyuanyu/anaconda3/lib/python3.11/site-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


ModuleNotFoundError: No module named 'sorl'

In [ ]:
ds = get_dataset("gsm8k", split="train", tokenizer=tokenizer, max_length=256)
B  = 16
ids  = torch.stack([ds[i]["input_ids"]      for i in range(B)]).to(device)
attn = torch.stack([ds[i]["attention_mask"] for i in range(B)]).to(device)
pl   = torch.tensor([ds[i]["prompt_len"]    for i in range(B)], device=device)
print(f"Batch: ids {ids.shape}")

## 1. Baseline — v6 vocab collapse

In [ ]:
def v6_select(model, ids, attn, pl, bv, N_ABS):
    with torch.no_grad():
        out = model(input_ids=ids, attention_mask=attn, output_hidden_states=True)
        h = out.hidden_states[-1]
    t_idx = (pl - 1).clamp(min=0, max=h.size(1) - 1)
    h_q = h[torch.arange(ids.size(0), device=device), t_idx]
    with torch.no_grad():
        logits_abs = model.model.lm_head(h_q)[:, bv:bv+N_ABS]
    return logits_abs.argmax(dim=-1)

N_BATCHES = 20
v6_counter = Counter()
for b0 in range(0, min(N_BATCHES * B, len(ds)), B):
    b1 = min(b0 + B, len(ds))
    bids  = torch.stack([ds[i]["input_ids"]      for i in range(b0, b1)]).to(device)
    battn = torch.stack([ds[i]["attention_mask"] for i in range(b0, b1)]).to(device)
    bpl   = torch.tensor([ds[i]["prompt_len"]    for i in range(b0, b1)], device=device)
    v6_counter.update(v6_select(model, bids, battn, bpl, bv, N_ABS).tolist())

total = sum(v6_counter.values())
ent   = -sum((c/total)*np.log(c/total+1e-9) for c in v6_counter.values())
print(f"v6: {len(v6_counter)}/{N_ABS} unique tokens | entropy={ent:.3f}/{np.log(N_ABS):.3f}")
print(f"Top-5: {v6_counter.most_common(5)}")

## 2. Frozen orthogonal codebook C

In [ ]:
def make_frozen_codebook(N_ABS, d, device, dtype=torch.float32):
    """Frozen orthogonal codebook. Rows = abstract token embeddings."""
    C = nn.Embedding(N_ABS, d)
    if N_ABS <= d:
        raw = torch.randn(d, N_ABS)
        Q, _ = torch.linalg.qr(raw)
        C.weight.data = Q.T
    else:
        raw = torch.randn(N_ABS, d)
        Q, _ = torch.linalg.qr(raw.T)
        C.weight.data[:d] = Q.T
        rest = torch.randn(N_ABS - d, d)
        C.weight.data[d:] = F.normalize(rest, dim=-1)
    C.weight.requires_grad_(False)
    return C.to(device=device, dtype=dtype)

C = make_frozen_codebook(N_ABS, d, device, dtype=model_dtype)

gram = (C.weight @ C.weight.T)
off_diag = gram.float() - torch.eye(N_ABS, device=device)
print(f"Codebook C: shape={C.weight.shape}, dtype={C.weight.dtype}, requires_grad={C.weight.requires_grad}")
print(f"Row norms: min={C.weight.float().norm(dim=1).min():.4f}, max={C.weight.float().norm(dim=1).max():.4f}")
print(f"Off-diagonal max (orthogonality check): {off_diag.abs().max():.6f}  (0 = perfect)")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
ax = axes[0]
im = ax.imshow(gram.detach().float().cpu().numpy(), cmap="RdBu_r", vmin=-1, vmax=1)
plt.colorbar(im, ax=ax)
ax.set_title("Gram matrix C @ C^T (should be ~identity)")
ax.set_xlabel("code j"); ax.set_ylabel("code i")

ax = axes[1]
im2 = ax.imshow(C.weight.detach().float().cpu().numpy()[:, :D_SHOW], aspect="auto", cmap="RdBu_r")
plt.colorbar(im2, ax=ax)
ax.set_title(f"Codebook C rows (first {D_SHOW} dims)")
ax.set_xlabel(f"Hidden dim (first {D_SHOW})"); ax.set_ylabel("Abstract token ID")
plt.tight_layout(); plt.show()

## 3. Learned projection W (identity init)

In [ ]:
def make_projection_W(d, device, dtype):
    W = nn.Linear(d, d, bias=False)
    nn.init.eye_(W.weight)
    return W.to(device=device, dtype=dtype)

W = make_projection_W(d, device, model_dtype)

eye = torch.eye(d, device=device, dtype=model_dtype)
print(f"W: {W.weight.shape} | dtype={W.weight.dtype} | params={W.weight.numel():,}")
print(f"||W - I||_F = {(W.weight - eye).float().norm():.6f}  (0 at init)")

CW = C.weight @ W.weight.T
gram_cw = (CW.float() @ CW.float().T).cpu()
print(f"||gram(CW) - I||_F at init = {(gram_cw - torch.eye(N_ABS)).norm():.6f}")

## 4. STE selection function

**Forward**: `multinomial` sampling with temperature (hard, discrete)  
**Backward**: same-temperature softmax proxy — gradient flows to W

In [ ]:
def select_abstract_token_ste(h, W, C, temperature=1.0):
    """
    h: (B, d)
    W: nn.Linear(d, d), learned, trained via STE
    C: nn.Embedding(N_ABS, d), frozen orthogonal
    temperature: controls sampling sharpness (forward) and gradient sharpness (backward)

    Returns: ste_embed (B,d), token_ids (B,), raw_logits (B,N_ABS)
    """
    h = h.to(dtype=W.weight.dtype)               # ensure dtype matches W
    Wh     = W(h)                                # (B, d)
    logits = Wh @ C.weight.T                     # (B, N_ABS), raw

    temp = max(float(temperature), 1e-10)

    # FORWARD: hard multinomial sample (same as extract_and_sample)
    with torch.no_grad():
        probs_hard = F.softmax(logits / temp, dim=-1)
        token_ids  = torch.multinomial(probs_hard, num_samples=1).squeeze(-1)  # (B,)
    hard_embed = C(token_ids)                    # (B, d), no grad

    # BACKWARD: soft proxy with grad
    soft_probs = F.softmax(logits / temp, dim=-1)  # (B, N_ABS), has grad via W
    soft_embed = soft_probs @ C.weight             # (B, d)

    # STE: value = hard sample, gradient = d(soft)/d(W)
    ste_embed = hard_embed + (soft_embed - soft_embed.detach())
    return ste_embed, token_ids, logits


# Sanity check
h_t = torch.randn(4, d, device=device, dtype=model_dtype)
W_t = make_projection_W(d, device, model_dtype)
ste_out, tids, _ = select_abstract_token_ste(h_t, W_t, C, temperature=1.0)
hard = C(tids)
print(f"||ste - hard|| = {(ste_out - hard).norm():.8f}  (should be 0)")
ste_out.sum().backward()
print(f"W grad is None? {W_t.weight.grad is None}  (False = grad flows ✓)")
print(f"||W.grad||_F = {W_t.weight.grad.float().norm():.4f}")

## 5. Forward pass integration via `inputs_embeds`

In [ ]:
from sorl.sorl_trainer import get_answer_start_index

def build_compressed_seq(ids, attn, pl, n_abs, bv, pad_id, answer_token_id=820):
    B = ids.size(0)
    ans_start = get_answer_start_index(ids, answer_token_id=answer_token_id)
    valid_len = attn.sum(dim=1)
    ans_len   = (valid_len - ans_start).clamp(min=0)
    max_len   = int((pl + n_abs + ans_len).max().item())
    comp_data = ids.new_full((B, max_len), pad_id)
    comp_attn = attn.new_zeros(B, max_len)
    for b in range(B):
        p, a, al = int(pl[b]), int(ans_start[b]), int(ans_len[b])
        comp_data[b, :p] = ids[b, :p];  comp_attn[b, :p] = 1
        comp_data[b, p:p+n_abs] = bv;   comp_attn[b, p:p+n_abs] = 1
        if al > 0:
            comp_data[b, p+n_abs:p+n_abs+al] = ids[b, a:a+al]
            comp_attn[b, p+n_abs:p+n_abs+al] = 1
    return comp_data, comp_attn, ans_start, pl + n_abs


def ste_forward_pass(model, ids, attn, pl, W, C, n_abs, bv, n_iter=3, temperature=1.0):
    B = ids.size(0)
    comp_data, comp_attn, _, ans_pos_s = build_compressed_seq(
        ids, attn, pl, n_abs, bv, tokenizer.pad_token_id)

    embed_layer  = model.model.model.embed_tokens
    safe_comp    = comp_data.clamp(max=bv - 1)
    inputs_embeds = embed_layer(safe_comp).clone()  # (B, L, d)

    iter_token_ids = []
    for it in range(n_iter):
        out = model.model.model(
            inputs_embeds=inputs_embeds,
            attention_mask=comp_attn,
            output_hidden_states=True,
            use_cache=False,
        )
        h = out.last_hidden_state  # (B, L, d)

        new_embeds     = inputs_embeds.clone()
        all_token_ids  = torch.zeros(B, n_abs, dtype=torch.long, device=device)

        for k in range(n_abs):
            pos_k    = pl + k                                                 # (B,)
            prev_pos = (pos_k - 1).clamp(min=0)
            h_prev   = h[torch.arange(B, device=device), prev_pos]           # (B, d)
            ste_emb, tok_ids, _ = select_abstract_token_ste(h_prev, W, C, temperature)
            all_token_ids[:, k] = tok_ids
            for b in range(B):
                new_embeds[b, pos_k[b], :] = ste_emb[b]

        inputs_embeds = new_embeds
        iter_token_ids.append(all_token_ids.cpu())

    return inputs_embeds, iter_token_ids, comp_attn, ans_pos_s


# Demo (no grad)
B_demo = 8
W_demo = make_projection_W(d, device, model_dtype)
with torch.no_grad():
    _, iter_tids, _, _ = ste_forward_pass(
        model, ids[:B_demo], attn[:B_demo], pl[:B_demo], W_demo, C, K, bv, n_iter=3)

print(f"STE forward: {len(iter_tids)} iters, K={K}")
for it, tids in enumerate(iter_tids):
    all_unique = len(set(tids.flatten().tolist()))
    print(f"  Iter {it}: {all_unique}/{N_ABS} unique tokens")

## 6. Vocab frequency: v6 vs STE (W=I)

In [ ]:
N_VIZ = 20
W_fresh = make_projection_W(d, device, model_dtype)
ste_counter_before = Counter()

for b0 in range(0, min(N_VIZ * 4, len(ds)), 4):
    b1   = min(b0 + 4, len(ds))
    bids = torch.stack([ds[i]["input_ids"]      for i in range(b0, b1)]).to(device)
    bat  = torch.stack([ds[i]["attention_mask"] for i in range(b0, b1)]).to(device)
    bpl  = torch.tensor([ds[i]["prompt_len"]    for i in range(b0, b1)], device=device)
    with torch.no_grad():
        _, tl, _, _ = ste_forward_pass(model, bids, bat, bpl, W_fresh, C, K, bv, n_iter=1)
    ste_counter_before.update(tl[0].flatten().tolist())

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, counter, title, color in zip(
    axes,
    [v6_counter, ste_counter_before],
    ["v6 (argmax)", "STE W=I (untrained)"],
    ["#c44e52", "#4c72b0"],
):
    cnts  = [counter.get(i, 0) for i in range(N_ABS)]
    tot   = sum(cnts)
    n_use = sum(1 for c in cnts if c > 0)
    ent   = -sum((c/tot)*np.log(c/tot+1e-9) for c in cnts if c > 0)
    ax.bar(range(N_ABS), cnts, color=color)
    ax.set_title(f"{title}\n{n_use}/{N_ABS} tokens | H={ent:.2f}")
    ax.set_xlabel("Abstract token ID")

# Codebook cosine sim
Cn      = F.normalize(C.weight, dim=1)
cos_mat = (Cn @ Cn.T).detach().float().cpu().numpy()
im = axes[2].imshow(cos_mat, cmap="RdBu_r", vmin=-1, vmax=1)
plt.colorbar(im, ax=axes[2])
axes[2].set_title("C cosine sim (ortho → diagonal)")
plt.tight_layout(); plt.show()

## 7. Mini training loop — only W trains

In [ ]:
W_trained = make_projection_W(d, device, model_dtype)
optim_W   = torch.optim.Adam(W_trained.parameters(), lr=1e-3)
eye_d     = torch.eye(d, device=device, dtype=model_dtype)

N_STEPS = 60
TRAIN_B = 4
N_ITER  = 2
losses, entropy_hist, drift = [], [], []

model.train()
for step in range(N_STEPS):
    b0   = (step * TRAIN_B) % (len(ds) - TRAIN_B)
    bids = torch.stack([ds[i]["input_ids"]      for i in range(b0, b0+TRAIN_B)]).to(device)
    bat  = torch.stack([ds[i]["attention_mask"] for i in range(b0, b0+TRAIN_B)]).to(device)
    bpl  = torch.tensor([ds[i]["prompt_len"]    for i in range(b0, b0+TRAIN_B)], device=device)

    comp_data_s, comp_attn_s, _, _ = build_compressed_seq(
        bids, bat, bpl, K, bv, tokenizer.pad_token_id)

    embed_layer   = model.model.model.embed_tokens
    inputs_embeds = embed_layer(comp_data_s.clamp(max=bv-1)).clone().detach()

    # Recursion
    for it in range(N_ITER):
        with torch.no_grad():
            h_all = model.model.model(
                inputs_embeds=inputs_embeds,
                attention_mask=comp_attn_s,
                output_hidden_states=True,
                use_cache=False,
            ).last_hidden_state
        new_embeds = inputs_embeds.clone().detach()
        for k in range(K):
            pos_k  = bpl + k
            h_prev = h_all[torch.arange(TRAIN_B, device=device), (pos_k-1).clamp(min=0)]
            ste_emb, _, _ = select_abstract_token_ste(h_prev, W_trained, C)
            for b in range(TRAIN_B):
                new_embeds[b, pos_k[b], :] = ste_emb[b]
        inputs_embeds = new_embeds

    # Final forward + traj loss
    logits_f = model.model.lm_head(
        model.model.model(inputs_embeds=inputs_embeds,
                          attention_mask=comp_attn_s, use_cache=False).last_hidden_state)

    lab = comp_data_s.clone().long()
    for b in range(TRAIN_B):
        lab[b, :int(bpl[b])+K]    = -100
        lab[b, lab[b] >= bv]      = -100
        lab[b, comp_attn_s[b]==0] = -100

    traj_loss = F.cross_entropy(
        logits_f[:, :-1].reshape(-1, logits_f.size(-1)),
        lab[:, 1:].reshape(-1), ignore_index=-100)

    optim_W.zero_grad(); traj_loss.backward(); optim_W.step()

    losses.append(traj_loss.item())
    drift.append((W_trained.weight - eye_d).float().norm().item())

    with torch.no_grad():
        h_s = h_all[torch.arange(TRAIN_B, device=device), (bpl-1).clamp(min=0)]
        _, _, lgts_s = select_abstract_token_ste(h_s, W_trained, C)
        p = F.softmax(lgts_s.float(), dim=-1).cpu().numpy()
        ent_s = float(-np.sum(p * np.log(p + 1e-9), axis=-1).mean())
    entropy_hist.append(ent_s)

    if (step + 1) % 10 == 0:
        print(f"step {step+1:3d} | loss={traj_loss.item():.4f} | H={ent_s:.3f} | ||W-I||={drift[-1]:.4f}")

model.eval()

## 8. Comparison: before vs after W training

In [ ]:
ste_counter_after = Counter()
for b0 in range(0, min(N_VIZ * 4, len(ds)), 4):
    b1   = min(b0 + 4, len(ds))
    bids = torch.stack([ds[i]["input_ids"]      for i in range(b0, b1)]).to(device)
    bat  = torch.stack([ds[i]["attention_mask"] for i in range(b0, b1)]).to(device)
    bpl  = torch.tensor([ds[i]["prompt_len"]    for i in range(b0, b1)], device=device)
    with torch.no_grad():
        _, tl, _, _ = ste_forward_pass(model, bids, bat, bpl, W_trained, C, K, bv, n_iter=1)
    ste_counter_after.update(tl[0].flatten().tolist())

fig, axes = plt.subplots(2, 3, figsize=(16, 8))

for ax, counter, title, color in zip(
    axes[0],
    [v6_counter, ste_counter_before, ste_counter_after],
    ["v6 (argmax C@h)", "STE W=I", f"STE trained ({N_STEPS} steps)"],
    ["#c44e52", "#4c72b0", "#55a868"],
):
    cnts  = [counter.get(i, 0) for i in range(N_ABS)]
    tot   = sum(cnts)
    n_use = sum(1 for c in cnts if c > 0)
    ent   = -sum((c/tot)*np.log(c/tot+1e-9) for c in cnts if c > 0)
    ax.bar(range(N_ABS), cnts, color=color)
    ax.set_title(f"{title}\n{n_use}/{N_ABS} tokens | H={ent:.2f}")
    ax.set_xlabel("Abstract token ID")

axes[1][0].plot(losses, color="#c44e52")
axes[1][0].set_title("Traj loss"); axes[1][0].set_xlabel("Step"); axes[1][0].grid(alpha=0.3)

axes[1][1].plot(entropy_hist, color="#4c72b0")
axes[1][1].axhline(np.log(N_ABS), color="k", ls="--", lw=0.8, label=f"max={np.log(N_ABS):.2f}")
axes[1][1].set_title("Token selection entropy"); axes[1][1].set_xlabel("Step")
axes[1][1].legend(); axes[1][1].grid(alpha=0.3)

axes[1][2].plot(drift, color="#dd8452")
axes[1][2].set_title("||W - I||_F"); axes[1][2].set_xlabel("Step"); axes[1][2].grid(alpha=0.3)

plt.tight_layout(); plt.show()

print("\n=== Summary ===")
for name, counter in [("v6", v6_counter), ("STE W=I", ste_counter_before), ("STE trained", ste_counter_after)]:
    cnts  = [counter.get(i, 0) for i in range(N_ABS)]
    tot   = sum(cnts)
    n_use = sum(1 for c in cnts if c > 0)
    ent   = -sum((c/tot)*np.log(c/tot+1e-9) for c in cnts if c > 0)
    print(f"  {name:20s}: {n_use:2d}/{N_ABS} tokens | H={ent:.3f}/{np.log(N_ABS):.3f}")

## 9. CW vs C: do codes stay spread after W trains?

In [ ]:
eye_d = torch.eye(d, device=device, dtype=model_dtype)

with torch.no_grad():
    CW_rows = C.weight @ W_trained.weight.T
    CW_norm = F.normalize(CW_rows, dim=1)
    C_norm  = F.normalize(C.weight, dim=1)
    cos_C  = (C_norm  @ C_norm.T).detach().float().cpu().numpy()
    cos_CW = (CW_norm @ CW_norm.T).detach().float().cpu().numpy()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, mat, title in zip(axes, [cos_C, cos_CW, cos_CW - cos_C],
                           ["C (frozen)", "CW (trained)", "Δ (CW - C)"]):
    vmax = 1.0 if title != "Δ (CW - C)" else 0.2
    im = ax.imshow(mat, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
    plt.colorbar(im, ax=ax)
    ax.set_title(f"Cosine sim: {title}")
plt.tight_layout(); plt.show()

mask = ~np.eye(N_ABS, dtype=bool)
print(f"C  off-diag: mean={cos_C[mask].mean():.4f}, max={cos_C[mask].max():.4f}")
print(f"CW off-diag: mean={cos_CW[mask].mean():.4f}, max={cos_CW[mask].max():.4f}")
print(f"||W - I||_F = {(W_trained.weight - eye_d).float().norm().item():.4f}")